<a href="https://colab.research.google.com/github/sweeden-ttu/profile/blob/stable/notebooks/usd-llm/03_megatron_bridge_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Scaling up with Megatron-Bridge (cluster)

This notebook is a **runbook**: it prepares the data on Colab and prints the commands to run on an 8×A100/H100 node. Megatron-LM's own docs now point HF↔Megatron conversion to **Megatron-Bridge**, which pins Megatron-Core as a submodule (`3rdparty/Megatron-LM`). Use the upstream pin unless you carry core patches in the `sweeden-ttu/Megatron-LM` fork.

Verified against Megatron-Bridge docs/PyPI (0.5.x–0.6.x) in October 2026; re-check names against the version you pin.

In [ ]:
!wget -q -O usdlab.py https://raw.githubusercontent.com/sweeden-ttu/profile/stable/notebooks/usd-llm/usdlab.py
!pip -q install usd-core
import usdlab as U, json
rows = U.make_sft_pairs(5000, seed=0)
test_fams = {'units', 'swap_label'}
with open('training.jsonl','w') as f:
    [f.write(json.dumps({'input': r['input'], 'output': r['output']})+'\n') for r in rows if r['family'] not in test_fams]
with open('validation.jsonl','w') as f:
    [f.write(json.dumps({'input': r['input'], 'output': r['output']})+'\n') for r in rows if r['family'] in test_fams]
!wc -l training.jsonl validation.jsonl

## On the cluster
```bash
# container: nvcr.io/nvidia/nemo:<tag>   or   pip install megatron-bridge   (Python 3.12)
git clone https://github.com/NVIDIA-NeMo/Megatron-Bridge && cd Megatron-Bridge

# 1. HF -> Megatron checkpoint (CPU is fine)
./scripts/conversion/convert.sh import --executor local --device cpu \
    --hf-model Qwen/Qwen3-8B --megatron-path ./ckpt/qwen3_8b
```

In [ ]:
RECIPE = '''
from megatron.bridge.recipes.qwen.qwen3 import qwen3_8b_peft_config   # LoRA, TP=1/PP=1, one 8-GPU node
cfg = qwen3_8b_peft_config(peft_scheme="lora")
cfg.checkpoint.pretrained_checkpoint = "./ckpt/qwen3_8b"
# cfg.dataset -> GPTSFTDatasetConfig(dataset_root="usd_sft/", seq_length=8192, ...)   # training.jsonl / validation.jsonl
# offline packing: enable_offline_packing=True, PackedSequenceSpecs(packed_sequence_size=8192)
'''
open('finetune_usd.py','w').write(RECIPE)
print('uv run python -m torch.distributed.run --nproc-per-node=8 finetune_usd.py')

## Export the adapter and evaluate
```bash
uv run python examples/conversion/adapter/export_adapter.py --hf-model-id Qwen/Qwen3-8B \
    --megatron-peft-checkpoint <ckpt> --output-hf-path ./usd_lora
uv run python examples/conversion/adapter/verify_adapter.py --hf-model-id Qwen/Qwen3-8B --hf-adapter-path ./usd_lora
```
Then serve `./usd_lora` with vLLM and score it with `usdlab.usdfid` exactly as in notebook 02.

| Recipe | TP / PP | Nodes |
|---|---|---|
| `qwen3_8b_peft_config` | 1 / 1 | 1 |
| `qwen3_32b_peft_config` | 1 / 1 + recompute | 1 |
| `qwen3_8b_sft_config` | 4 / 1 | 1 |
| `qwen3_32b_sft_config` | 8 / 2 | 2 |